In [ ]:
import os
import json
import random
import hashlib
from typing import Dict, List

from datasets import load_dataset

OUT_DIR = "/root/data/eval_data"
SEED = 0

# None = 全量导出；调试时可改小
N_HELLASWAG = None
N_ARC = None
N_WINO = None
N_TQA = None

random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)


def write_jsonl(path: str, records: List[Dict]):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def sha1_text(x: str) -> str:
    return hashlib.sha1(x.encode("utf-8")).hexdigest()


def maybe_subsample(records: List[Dict], n: int):
    if n is None or n >= len(records):
        return records
    idxs = list(range(len(records)))
    random.shuffle(idxs)
    idxs = idxs[:n]
    return [records[i] for i in idxs]


def choice_letters(k: int) -> List[str]:
    return [chr(ord("A") + i) for i in range(k)]


def build_mc_prompt(question: str, choices: List[str]) -> str:
    """
    统一的选择题评测口径：
    Question: ...
    A. ...
    B. ...
    ...
    
    Answer: 
    """
    lines = [f"Question: {question.strip()}"]
    for i, c in enumerate(choices):
        lines.append(f"{chr(ord('A') + i)}. {str(c).strip()}")
    lines.append("")
    lines.append("Answer: ")
    return "\n".join(lines)


def make_record(task: str, source_index: int, question: str, choices: List[str], answer_idx: int, split: str, extra_meta: Dict = None):
    prompt_text = build_mc_prompt(question, choices)
    answer_letter = chr(ord("A") + answer_idx)

    rec = {
        "id": sha1_text(f"{task}|||{source_index}|||{question[:200]}"),
        "task": task,
        "prompt_text": prompt_text,
        "choices": choices,
        "answer_idx": answer_idx,
        "answer_letter": answer_letter,
        "num_choices": len(choices),
        "meta": {
            "source_index": source_index,
            "split": split,
        }
    }
    if extra_meta:
        rec["meta"].update(extra_meta)
    return rec


# =========================================================
# 1) HellaSwag
# =========================================================
def build_hellaswag():
    ds = load_dataset("hellaswag", split="validation")
    records = []

    for i, ex in enumerate(ds):
        ctx_a = (ex.get("ctx_a") or "").strip()
        ctx_b = (ex.get("ctx_b") or "").strip()
        activity_label = (ex.get("activity_label") or "").strip()
        endings = ex.get("endings", [])
        label = ex.get("label")

        if not endings or label is None:
            continue

        question_parts = []
        if activity_label:
            question_parts.append(f"Activity: {activity_label}")
        if ctx_a:
            question_parts.append(ctx_a)
        if ctx_b:
            question_parts.append(ctx_b)

        question = "\n".join(question_parts).strip()
        if not question:
            continue

        try:
            answer_idx = int(label)
        except Exception:
            continue

        choices = [str(x).strip() for x in endings]
        if not (0 <= answer_idx < len(choices)):
            continue

        records.append(
            make_record(
                task="hellaswag",
                source_index=i,
                question=question,
                choices=choices,
                answer_idx=answer_idx,
                split="validation",
                extra_meta={"raw_label": str(label)}
            )
        )

    records = maybe_subsample(records, N_HELLASWAG)
    write_jsonl(os.path.join(OUT_DIR, "hellaswag.jsonl"), records)
    print("hellaswag:", len(records))


# =========================================================
# 2) ARC-Challenge
# =========================================================
def build_arc():
    ds = load_dataset("ai2_arc", "ARC-Challenge", split="validation")
    records = []

    label_to_idx_fallback = {chr(ord("A") + i): i for i in range(10)}
    label_to_idx_fallback.update({str(i + 1): i for i in range(10)})

    for i, ex in enumerate(ds):
        question = (ex.get("question") or "").strip()
        choices_obj = ex.get("choices", {})
        choice_texts = choices_obj.get("text", [])
        choice_labels = choices_obj.get("label", [])
        answer_key = str(ex.get("answerKey", "")).strip()

        if not question or not choice_texts or not answer_key:
            continue
        if len(choice_texts) != len(choice_labels):
            continue

        label_map = {str(lab).strip(): j for j, lab in enumerate(choice_labels)}

        if answer_key in label_map:
            answer_idx = label_map[answer_key]
        elif answer_key in label_to_idx_fallback and label_to_idx_fallback[answer_key] < len(choice_texts):
            answer_idx = label_to_idx_fallback[answer_key]
        else:
            continue

        choices = [str(x).strip() for x in choice_texts]

        records.append(
            make_record(
                task="arc_challenge",
                source_index=i,
                question=question,
                choices=choices,
                answer_idx=answer_idx,
                split="validation",
                extra_meta={"raw_answer_key": answer_key}
            )
        )

    records = maybe_subsample(records, N_ARC)
    write_jsonl(os.path.join(OUT_DIR, "arc_challenge.jsonl"), records)
    print("arc_challenge:", len(records))


# =========================================================
# 3) WinoGrande
# =========================================================
def build_winogrande():
    ds = load_dataset("winogrande", "winogrande_xl", split="validation")
    records = []

    for i, ex in enumerate(ds):
        sentence = (ex.get("sentence") or "").strip()
        option1 = (ex.get("option1") or "").strip()
        option2 = (ex.get("option2") or "").strip()
        answer = str(ex.get("answer", "")).strip()

        if not sentence or not option1 or not option2 or answer not in {"1", "2"}:
            continue

        answer_idx = int(answer) - 1
        choices = [option1, option2]

        records.append(
            make_record(
                task="winogrande",
                source_index=i,
                question=sentence,
                choices=choices,
                answer_idx=answer_idx,
                split="validation",
                extra_meta={}
            )
        )

    records = maybe_subsample(records, N_WINO)
    write_jsonl(os.path.join(OUT_DIR, "winogrande.jsonl"), records)
    print("winogrande:", len(records))


# =========================================================
# 4) TruthfulQA-MC
# =========================================================
def build_truthfulqa_mc():
    ds = load_dataset("truthful_qa", "multiple_choice", split="validation")
    records = []

    for i, ex in enumerate(ds):
        question = (ex.get("question") or "").strip()
        mc1_targets = ex.get("mc1_targets", {})
        choices = mc1_targets.get("choices", [])
        labels = mc1_targets.get("labels", [])

        if not question or not choices or not labels or len(choices) != len(labels):
            continue

        correct = [j for j, x in enumerate(labels) if int(x) == 1]
        if len(correct) != 1:
            continue

        answer_idx = correct[0]
        choices = [str(x).strip() for x in choices]

        records.append(
            make_record(
                task="truthfulqa_mc",
                source_index=i,
                question=question,
                choices=choices,
                answer_idx=answer_idx,
                split="validation",
                extra_meta={}
            )
        )

    records = maybe_subsample(records, N_TQA)
    write_jsonl(os.path.join(OUT_DIR, "truthfulqa_mc.jsonl"), records)
    print("truthfulqa_mc:", len(records))


# =========================================================
# 5) Manifest
# =========================================================
def build_manifest():
    manifest = [
        {"name": "hellaswag", "path": os.path.join(OUT_DIR, "hellaswag.jsonl")},
        {"name": "arc_challenge", "path": os.path.join(OUT_DIR, "arc_challenge.jsonl")},
        {"name": "winogrande", "path": os.path.join(OUT_DIR, "winogrande.jsonl")},
        {"name": "truthfulqa_mc", "path": os.path.join(OUT_DIR, "truthfulqa_mc.jsonl")},
    ]
    with open(os.path.join(OUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print("manifest:", os.path.join(OUT_DIR, "manifest.json"))


# =========================================================
# Run all
# =========================================================
build_hellaswag()
build_arc()
build_winogrande()
build_truthfulqa_mc()
build_manifest()

print("\nDone. Files written to:", OUT_DIR)